# Day 2 Football detection, tracking and teams

This is a reconstructed V4-equivalent candidate, pending comparison with the original outputs. Start a fresh T4 runtime. Upload sports_vision_v4_equivalent.zip to /content/. Setup extracts the candidate and links the Drive video. Run the 15-second validation first; each trial automatically compares its existing exports with the original V4 run.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from zipfile import ZipFile

from google.colab import drive

ARCHIVE = Path('/content/sports_vision_v4_equivalent.zip')
ROOT = Path('/content/sports-vision-v4-equivalent')
DRIVE_ROOT = Path('/content/drive/MyDrive/Sports Vision Tracking')
FOOTBALL = DRIVE_ROOT / 'Data/Football.mp4'
OUTPUTS = DRIVE_ROOT / 'outputs/football'
if ROOT.exists():
    raise RuntimeError(f'{ROOT} already exists. Start a fresh T4 runtime and rerun setup.')
assert ARCHIVE.is_file(), f'Upload the project ZIP to {ARCHIVE}'
gpus = subprocess.check_output(['nvidia-smi', '-L'], text=True)
assert 'T4' in gpus, f'Select a T4 GPU runtime. Detected GPUs: {gpus.strip()}'
drive.mount('/content/drive')
assert FOOTBALL.is_file(), f'Football video missing at {FOOTBALL}'

with ZipFile(ARCHIVE) as bundle:
    required = {'requirements.txt', 'scripts/track_football.py',
                'configs/football_config.yaml', 'src/common/video_loader.py'}
    missing = required.difference(bundle.namelist())
    if missing:
        raise ValueError(f'Project ZIP is missing {sorted(missing)}')
    bundle.extractall(ROOT)

video_link = ROOT / 'Data/Football.mp4'
video_link.parent.mkdir(parents=True, exist_ok=True)
video_link.symlink_to(FOOTBALL)
assert video_link.is_symlink() and video_link.resolve(strict=True) == FOOTBALL.resolve(strict=True)
requirements = ROOT / 'requirements.txt'
assert requirements.is_file() and (ROOT / 'scripts/track_football.py').is_file()
OUTPUTS.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print(f'Project: {ROOT}\nFootball video: {video_link} -> {FOOTBALL}\nOutputs: {OUTPUTS}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)

In [ ]:
import torch

assert torch.cuda.is_available(), 'Switch this notebook to a GPU runtime and rerun setup'
print(torch.cuda.get_device_name(0))

def run_trial(name, seconds, start_frame, ball_fallback=False):
    from datetime import datetime, timezone
    output = OUTPUTS / (name + '_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
    command = [
        sys.executable, 'scripts/track_football.py',
        '--device', 'cuda:0', '--start-frame', str(start_frame),
        '--seconds', str(seconds), '--output-dir', str(output),
    ]
    if ball_fallback:
        command.append('--ball-fallback')
    subprocess.run(command, check=True)
    metrics = json.loads((output / 'metrics.json').read_text())
    assert metrics['frames_processed'] == round(seconds * 25)
    for artifact in ('annotated.mp4', 'tracks.json', 'tracks.csv', 'pitch_diagnostics.mp4', 'pitch_diagnostics.jsonl'):
        assert (output / artifact).stat().st_size > 0
    print(f'Artifacts saved to {output}')
    original_name = 'smoke_v4_20261106_150141_706053' if seconds == 15 else 'one_minute_v4_20261106_150643_440214'
    if seconds in (15, 60) and start_frame == 1500:
        subprocess.run([sys.executable, 'scripts/compare_v4.py', '--original', str(OUTPUTS / original_name), '--candidate', str(output)], check=True)
    return metrics

## 15-second smoke trial

Inspect comparison.json, annotated.mp4, and pitch_diagnostics.mp4 before running the minute. The comparison records exact per-frame differences in IDs, labels, boxes, ball detections, CSV records, and pitch decisions. Matching summaries alone do not establish equivalence.

In [ ]:
smoke_metrics = run_trial('smoke_v4_equivalent', seconds=15, start_frame=1500, ball_fallback=True)

## Configurable one-minute trial

Use the same start and duration with the fallback disabled if you want to compare throughput and ball detection coverage. Inspect ball detections visually for false positives.

In [ ]:
START_FRAME = 1500
SECONDS = 60
BALL_FALLBACK = True

trial_metrics = run_trial('one_minute_v4_equivalent', SECONDS, START_FRAME, BALL_FALLBACK)